In [1]:
from tool_server.utils.utils import *
from tool_server.utils.prompts import tool_planning_model_prompt_one_tool_call
from tqdm import tqdm
import json, os


model_name_dict = {
    "qwen25vl3b": "Qwen-2.5VL-3B",
    "qwen25vl3b_sftv1": "Qwen-2.5VL-3B SFT v1",
    # "qwen25vl3b_notool": "Qwen-2.5VL-3B /wo Tools",
    "qwen25vl7b": "Qwen-2.5VL-7B",
    "qwen25vl7b_sftv1": "Qwen-2.5VL-7B SFT v1",
    # "qwen25vl7b_notool": "Qwen-2.5VL-7B /wo Tools",
}
task_list = ["chartqa", "vstar"]
res_base_dir = "/mnt/petrelfs/songmingyang/code/reasoning/tool-agent/tool_server/tf_eval/scripts/logs/ckpt"


In [2]:
def get_statistic_dict(model_name_dict,res_base_dir):
    statistic_dict = {}
    for model_name, full_name in model_name_dict.items():
        current_dir = os.path.join(res_base_dir, model_name)
        benchmark_ckpts = os.listdir(current_dir)
        model_ckpt_dict = {}
        for benchmark_ckpt in tqdm(benchmark_ckpts, desc=f"Processing {full_name}"):
            if not benchmark_ckpt.endswith(".jsonl"):
                continue
            benchmark_ckpt_path = os.path.join(current_dir, benchmark_ckpt)
            benchmark_ckpt_res = process_jsonl(benchmark_ckpt_path)
            benchmark_ckpt_res = [k["results"]["results"] for k in benchmark_ckpt_res]
            benchmark_name = benchmark_ckpt.split(".jsonl")[0]
            model_ckpt_dict[benchmark_name] = benchmark_ckpt_res
        statistic_dict[full_name] = model_ckpt_dict
    return statistic_dict

def get_str(statistic_dict,benchmark_list = ["chartqa", "vstar"]):
    res_str = ""
    for model_name, benchmark_dict in statistic_dict.items():
        res_str += f"\\textbf{{{model_name}}}"
        for benchmark_name in benchmark_list:
            rate_res = get_tool_statistic_by_benchmark_item(benchmark_dict, benchmark_name)
            if not rate_res:
                res_str += f" & - & - & - "
            else:
                for k,v in rate_res.items():
                    res_str += f" & {v:.2f}"
        res_str += "\\\\\n"
    
    return res_str

def get_tool_statistic_by_benchmark_item(ckpt_list_dict, benchmark_name):
    if benchmark_name not in ckpt_list_dict:
        return None
    benchmark_ckpt_list = ckpt_list_dict[benchmark_name]
    
    tool_try_num_list = []
    tool_qualified_cfg_num_list = []
    tool_succ_num_list = []
    for idx,item in enumerate(benchmark_ckpt_list):
        tool_config = item["tool_cfg"]
        tool_response = item["tool_response"]
        tool_try_num = len(tool_config)
        tool_qualified_cfg_num = len([k for k in tool_config if k is not None])
        
        tool_succ_num = 0
        for response in tool_response:
            try:
                # if response:
                #     print("error_code" in response)
                # if isinstance(response, str):
                #     response = json.loads(response)
                error_code = response.get("error_code", None)
                if error_code == 0:
                    tool_succ_num += 1
            except:
                pass
        tool_try_num_list.append(tool_try_num)
        tool_qualified_cfg_num_list.append(tool_qualified_cfg_num)
        tool_succ_num_list.append(tool_succ_num)
    
    average_tool_try_num = sum(tool_try_num_list) / len(tool_try_num_list) if tool_try_num_list else 0
    average_tool_config_qualified_rate = sum(tool_qualified_cfg_num_list) / sum(tool_try_num_list) if sum(tool_try_num_list) > 0 else 0
    average_tool_succ_rate = sum(tool_succ_num_list) / sum(tool_qualified_cfg_num_list) if sum(tool_qualified_cfg_num_list) > 0 else 0
    return {
        "average_tool_try_num": average_tool_try_num,
        "average_tool_config_qualified_rate": average_tool_config_qualified_rate,
        "average_tool_succ_rate": average_tool_succ_rate
    }
            
                    

In [3]:
res_dict = get_statistic_dict(model_name_dict, res_base_dir)


Processing Qwen-2.5VL-7B SFT v1: 100%|██████████| 8/8 [00:39<00:00,  4.93s/it]


In [4]:
res_str = get_str(res_dict, task_list)
print(res_str)

\textbf{Qwen-2.5VL-3B} & 0.12 & 0.42 & 0.84 & 4.32 & 0.02 & 0.94\\
\textbf{Qwen-2.5VL-3B SFT v1} & 1.90 & 0.91 & 0.83 & 2.47 & 0.74 & 0.93\\
\textbf{Qwen-2.5VL-7B} & 0.12 & 0.75 & 0.96 & 0.98 & 0.01 & 1.00\\
\textbf{Qwen-2.5VL-7B SFT v1} & 2.00 & 0.85 & 0.82 & 3.01 & 0.59 & 0.89\\

